In [1]:
import os
import google.generativeai as genai
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Configure the Gemini API with your Free Key
API_KEY = "ADD_YOUR_API"
genai.configure(api_key=API_KEY)

# 2. Initialize your exact AraBERT Embedding Function
print("Loading AraBERT Embedding model...")
model_name = "aubmindlab/bert-base-arabertv02"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'} 
)

# 3. Connect to your existing saved Chroma Database
print("Connecting to Chroma Vector Store...")
db = Chroma(
    embedding_function=embeddings,
    collection_name="my_rag_project",
    persist_directory="./my_vector_db"
)


def ask_chroma_books_stream(query):
    try:
        # Retrieve the top 4 closest matching text chunks
        docs = db.similarity_search(query, k=4)
        
        if not docs:
            print("\nالإجابة: المعلومة غير متوفرة في الكتب المرفقة.")
            return
            
        context_text = "\n\n".join([doc.page_content for doc in docs])
        
        try:
            file_source = os.path.basename(docs[0].metadata.get('source', 'الكتب المخزنة'))
        except Exception:
            file_source = "الكتب المخزنة"

        # Balanced prompt allowing natural phrasing reasoning
        system_instruction = (
            "أنت مساعد ذكي ومتخصص في تحليل النصوص العربية ومحدد جداً. مهمتك هي الإجابة على سؤال المستخدم بناءً على السياق المستخرج المقدم فقط.\n"
            "شروط الإجابة:\n"
            "1. يجب أن تعتمد إجابتك بالكامل على السياق المرفق أدناه.\n"
            "2. لا تضف أي معلومات خارجية تماماً من خارج هذا النص.\n"
            "3. يمكنك فهم المعنى المرادف لغوياً بدقة.\n"
            "4. إذا كان النص المرفق لا يحتوي على إجابة مباشرة، قل فقط: 'المعلومة غير متوفرة في الكتب المرفقة.' دون زيادة."
        )
        
        user_prompt = f"""
{system_instruction}

[السياق المستخرج من - المصدر: {file_source}]:
{context_text}

[سؤال المستخدم]:
{query}

الإجابة المباشرة:
"""

        # 4. Call the model and set stream=True
        model = genai.GenerativeModel('gemini-3.6-flash')
        response = model.generate_content(user_prompt, stream=True)
        
        print("\nالإجابة: ", end="", flush=True)
        
        # Loop through chunks and print them immediately to your notebook output
        for chunk in response:
            print(chunk.text, end="", flush=True)
        print("\n") # New line after generation completes

    except Exception as e:
        print(f"\nحدث خطأ أثناء المعالجة: {str(e)}")


# 5. Interactive loop to chat with your books
if __name__ == "__main__":
    print("\n==== محرك البث الفوري جاهز تماماً! ====")
    print("يمكنك البدء بطرح الأسئلة (اكتب 'خروج' للإنهاء):")
    
    while True:
        user_query = input("\nسؤالك باللغة العربية: ")
        if user_query.strip() in ["خروج", "exit", "quit"]:
            print("تم إغلاق البرنامج بنجاح.")
            break
            
        if not user_query.strip():
            continue
            
        print("جاري البحث دلالياً...")
        ask_chroma_books_stream(user_query)


ModuleNotFoundError: No module named 'google.generativeai'